# Mahjong Tile Detection: YOLOv8 Training Loop

This notebook outlines the process of fetching incorrect detection reports from Firebase Storage, adding them to the dataset, retraining a YOLOv8 model, and exporting it to ONNX for use in the web application.

In [ ]:
!pip install ultralytics firebase-admin roboflow

## 1. Fetching Data from Firebase Storage

In [ ]:
import firebase_admin
from firebase_admin import credentials, storage
import json
import os
import urllib.request

# Initialize Firebase Admin (Requires serviceAccountKey.json)
cred = credentials.Certificate('path/to/serviceAccountKey.json')
firebase_admin.initialize_app(cred, {
    'storageBucket': 'your-firebase-project.appspot.com'
})

bucket = storage.bucket()

# Create directories for raw data
os.makedirs('dataset/raw_images', exist_ok=True)
os.makedirs('dataset/raw_annotations', exist_ok=True)

def download_new_data():
    blobs = bucket.list_blobs(prefix='dataset-collection/annotations/')
    for blob in blobs:
        if blob.name.endswith('.json'):
            # Download annotation
            annotation_data = json.loads(blob.download_as_string())
            image_id = annotation_data['imageId']
            image_url = annotation_data['imageUrl']
            
            # Save annotation locally
            with open(f'dataset/raw_annotations/{image_id}.json', 'w') as f:
                json.dump(annotation_data, f)
                
            # Download corresponding image
            urllib.request.urlretrieve(image_url, f'dataset/raw_images/{image_id}.jpg')
            
            print(f"Downloaded {image_id}")

download_new_data()

## 2. Combine with Roboflow Dataset

We combine our newly collected dataset with the base Mahjong dataset from Roboflow.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("jon-chan-gnsoa").project("mahjong-baq4s")
version = project.version(1)
dataset = version.download("yolov8")

print("Base dataset downloaded to:", dataset.location)

## 3. Train YOLOv8 Model

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8n model
model = YOLO('yolov8n.pt')

# Train the model on the combined dataset (ensure data.yaml is updated to include new data)
results = model.train(data=f'{dataset.location}/data.yaml', epochs=100, imgsz=640)


## 4. Export to ONNX
Export the trained model to ONNX format for use with `onnxruntime-web` in the React frontend.

In [ ]:
# Export the model
success = model.export(format='onnx', imgsz=640, dynamic=True)

print(f"Model exported successfully to: {success}")